# Transformer Testing

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from modellens import ModelLens


# Load model
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

/opt/anaconda3/envs/modellens/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7872.73it/s]


In [2]:
# Create lens
lens = ModelLens(model)
lens.adapter.set_tokenizer(tokenizer)

# Check summary
print(lens)
print(lens.summary())

ModelLens(
  backend=huggingface,
  architecture=transformer,
  params=124,439,808,
  analyses=['activation_patching', 'attention_maps', 'embeddings', 'hooks', 'layer_probing', 'residual_stream']
)
{'backend': 'huggingface', 'architecture': 'transformer', 'total_parameters': 124439808, 'layer_count': 163, 'hooks_attached': 0, 'activations_captured': [], 'available_analyses': ['activation_patching', 'attention_maps', 'embeddings', 'hooks', 'layer_probing', 'residual_stream'], 'unavailable_analyses': ['feature_map_analysis', 'filter_analysis', 'gate_analysis']}


In [3]:
from modellens.analysis.layer_evolution import run_layer_evolution, summarize_evolution

inputs = tokenizer("The capital of France is", return_tensors="pt")
results = run_layer_evolution(lens, inputs, top_k=5, tokenizer=tokenizer)
print(summarize_evolution(results))

Layer evolution: 113 layers analyzed
Position: 4

First confidence: transformer.h.0.attn.c_proj (entropy: 1.147)
Biggest shift: transformer.h.11 (KL: 27.095)
Top-1 stabilizes after: transformer.h.11

Top token trajectories (final probability):
   the            | start: 0.0000 -> final: 0.0846 | peak: 1.0000 ↑
   now            | start: 0.0000 -> final: 0.0479 | peak: 0.0846 ↑
   a              | start: 0.0000 -> final: 0.0462 | peak: 0.2907 ↑
   France         | start: 0.0000 -> final: 0.0324 | peak: 0.1788 ↑
   Paris          | start: 0.0000 -> final: 0.0322 | peak: 0.0474 ↑


In [3]:
from modellens.analysis.circuit_discovery import discover_circuit, summarize_circuit

clean = tokenizer("The Eiffel Tower is in", return_tensors="pt")
corrupted = tokenizer("The Colosseum is in", return_tensors="pt")
circuit = discover_circuit(lens, clean, corrupted, importance_threshold=0.1)

print(summarize_circuit(circuit))

Circuit: 19 components, 24 connections
Clean metric: -69.1741
Corrupted metric: -70.8887
Total effect: -1.7146

CRITICAL (8):
  transformer.h.1.attn (attention) — effect: -0.703
  transformer.h.3.mlp (mlp) — effect: +1.671
  transformer.h.5.mlp (mlp) — effect: +2.551
  transformer.h.6.mlp (mlp) — effect: +2.126
  transformer.h.7.attn (attention) — effect: +1.512
  transformer.h.9.mlp (mlp) — effect: +2.072
  transformer.h.10.mlp (mlp) — effect: +0.846
  transformer.h.11.attn (attention) — effect: +0.962

BOOSTER (2):
  transformer.h.3.attn (attention) — effect: -0.458
  transformer.h.4.mlp (mlp) — effect: -0.368

GATE (2):
  transformer.h.10.attn (attention) — effect: +0.301
  transformer.h.11.mlp (mlp) — effect: +0.156

PROCESSOR (7):
  transformer.h.0.attn (attention) — effect: -0.199
  transformer.h.0.mlp (mlp) — effect: +0.572
  transformer.h.1.mlp (mlp) — effect: +0.310
  transformer.h.2.mlp (mlp) — effect: +0.382
  transformer.h.4.attn (attention) — effect: +0.675
  transformer.h

In [3]:
# Tokenize input
inputs = tokenizer("The capital of France is", return_tensors="pt")

# Logit lens / layer probe
results = lens.layer_probe(inputs, top_k=5)
from modellens.analysis.logit_lens import decode_logit_lens
decoded = decode_logit_lens(results, tokenizer=tokenizer)
for layer, preds in list(decoded.items())[:3]:
    print(f"{layer}: {preds}")

transformer.wte: [(' is', 0.0035959226079285145), (' has', 0.0011945082806050777), (' was', 0.00097653764532879), (' does', 0.0005770876887254417), (' are', 0.0005422612302936614)]
transformer.wpe: [('theless', 8.032188634388149e-05), ('Thumbnail', 7.765968621242791e-05), ('lust', 7.400718459393829e-05), ('מ', 7.280485442606732e-05), (' gaping', 7.028273103060201e-05)]
transformer.drop: [(' is', 0.003390722908079624), (' has', 0.0012133164564147592), (' was', 0.000962700869422406), (' does', 0.0006336725200526416), (' will', 0.0006186149548739195)]


In [4]:
# Attention maps
attn = lens.attention_map(inputs)
print(f"Attention layers: {attn['num_layers']}")

Attention layers: 12


In [5]:
# Residual stream
residual = lens.residual_stream(inputs)
print(f"Layers analyzed: {residual['num_layers_analyzed']}")

Layers analyzed: 11


In [7]:
# Embeddings
emb = lens.embeddings(inputs)
print(f"Embedding dim: {emb['embed_dim']}")
print(f"Seq length: {emb['seq_length']}")
print(f"Similarity matrix shape: {emb['similarity_matrix'].shape}")

Embedding dim: 768
Seq length: 5
Similarity matrix shape: torch.Size([5, 5])


# CNN Testing

In [10]:
import torch
import torch.nn as nn
from modellens import ModelLens

# Simple CNN
class TestCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(32, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.flatten(1)
        return self.classifier(x)

cnn = TestCNN()
lens = ModelLens(cnn)

print(lens)
print(f"\nAvailable: {lens.available_analyses()}")

ModelLens(
  backend=pytorch,
  architecture=convolutional,
  params=5,130,
  analyses=['activation_patching', 'feature_map_analysis', 'filter_analysis', 'hooks', 'layer_probing']
)

Available: ['activation_patching', 'feature_map_analysis', 'filter_analysis', 'hooks', 'layer_probing']


In [11]:
# Test filter analysis
img = torch.randn(1, 1, 28, 28)
filters = lens.filter_analysis(img)
print(f"\nTotal filters: {filters['total_filters']}")
print(f"Dead filters: {filters['total_dead_filters']}")


Total filters: 48
Dead filters: 0


In [12]:
# Test feature map analysis
fmaps = lens.feature_maps(img)
print(f"\nLayers tracked: {fmaps['num_layers_tracked']}")
print(f"Spatial reduction: {fmaps['spatial_reduction']}x")
print(f"Channel expansion: {fmaps['channel_expansion']}x")


Layers tracked: 5
Spatial reduction: 28.0x
Channel expansion: 0.625x


In [13]:
for entry in fmaps['evolution']:
    print(f"{entry['layer']}: channels={entry['channels']}, spatial={entry['spatial_h']}x{entry['spatial_w']}")

conv1: channels=16, spatial=28x28
pool1: channels=16, spatial=14x14
conv2: channels=32, spatial=14x14
pool2: channels=32, spatial=1x1
classifier: channels=10, spatial=1x1


# LSTM Testing

In [14]:
import torch
import torch.nn as nn
from modellens import ModelLens

class TestLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(100, 32)
        self.lstm = nn.LSTM(32, 64, batch_first=True, num_layers=2)
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

lstm_model = TestLSTM()
lens = ModelLens(lstm_model)

print(lens)
print(f"\nAvailable: {lens.available_analyses()}")

ModelLens(
  backend=pytorch,
  architecture=recurrent,
  params=62,218,
  analyses=['activation_patching', 'embeddings', 'gate_analysis', 'hooks', 'layer_probing']
)

Available: ['activation_patching', 'embeddings', 'gate_analysis', 'hooks', 'layer_probing']


In [15]:
# Run gate analysis
tokens = torch.randint(0, 100, (1, 15))  # batch of 1, sequence length 15
gates = lens.gate_analysis(tokens)

result = gates["layer_results"]["lstm"]
print(f"\nType: {result['type']}")
print(f"Hidden size: {result['hidden_size']}")
print(f"Num layers: {result['num_layers']}")
print(f"Hidden norm trend: {result['hidden_evolution']['norm_trend']}")
print(f"Max change at timestep: {result['hidden_evolution']['max_delta_timestep']}")


Type: lstm
Hidden size: 64
Num layers: 2
Hidden norm trend: increasing
Max change at timestep: 1


In [16]:
# Gate weight stats
for layer_stat in result["gate_weight_stats"]:
    layer = layer_stat["layer"]
    print(f"\nLayer {layer}:")
    for gate in ["input", "forget", "cell", "output"]:
        stats = layer_stat[gate]
        print(f"  {gate}: ih_norm={stats['input_weight_norm']:.3f}, hh_norm={stats['hidden_weight_norm']:.3f}")


Layer 0:
  input: ih_norm=3.253, hh_norm=4.633
  forget: ih_norm=3.307, hh_norm=4.663
  cell: ih_norm=3.306, hh_norm=4.651
  output: ih_norm=3.245, hh_norm=4.673

Layer 1:
  input: ih_norm=4.652, hh_norm=4.620
  forget: ih_norm=4.606, hh_norm=4.621
  cell: ih_norm=4.620, hh_norm=4.556
  output: ih_norm=4.683, hh_norm=4.672
